# Lecture 8 · Notebook 2: Tokenizers and Models

Natural Language Processing (2026), SeoulTech.

`AutoTokenizer` and `AutoModel*` load any checkpoint on the Hub by its name.
We look at what the tokenizer produces, what the model returns, and where the loss is computed.

*Runtime: CPU is enough.*

In [1]:
# Run this cell first. Colab already has PyTorch; this installs the tested library versions.
%pip install -q "transformers>=5.20,<6" "datasets>=5.1,<6" "accelerate>=1.10"

import torch, transformers, datasets
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 5.21.1 | datasets 5.1.0 | torch 2.15.0
device: mps 


## 1. Text → token ids → text

In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# bert 모델이 학습했던 tokenizer을 그대로 가지고옴
enc = bert_tok("Hello world")
print(enc) #tokenizer로 가지고 오는얘                                            # input_ids, token_type_ids<문장 인덱스임>, attention_mask
print(bert_tok.convert_ids_to_tokens(enc["input_ids"]))
print(bert_tok.decode(enc["input_ids"]))
print(bert_tok.decode(enc["input_ids"], skip_special_tokens=True))

{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}
['[CLS]', 'hello', 'world', '[SEP]']
[CLS] hello world [SEP]
hello world


The steps one by one: `tokenize` (text → subword strings), `convert_tokens_to_ids`
(strings → ids), and the special tokens `[CLS]` and `[SEP]` that BERT expects (Lecture 6).

In [ ]:
text = "Tokenization splits unbelievable words."
tokens = bert_tok.tokenize(text) #text를 토큰화
ids = bert_tok.convert_tokens_to_ids(tokens) # token을 임베딩으로 
print(tokens)
print(ids)
print([bert_tok.cls_token_id] + ids + [bert_tok.sep_token_id] == bert_tok(text)["input_ids"])

['token', '##ization', 'splits', 'unbelievable', 'words', '.']
[19204, 3989, 19584, 23653, 2616, 1012]
True


## 2. Same text, different tokenizers

BERT uses WordPiece (`##` marks a piece inside a word). GPT-2 and SmolLM2 use byte-level BPE
(`Ġ` marks a space before the token, as in Homework 1). Qwen2.5 was trained on much more non-English text.

In [ ]:
names = {
    "BERT (WordPiece)":      "google-bert/bert-base-uncased",
    "GPT-2 (byte BPE)":      "openai-community/gpt2",
    "SmolLM2 (byte BPE)":    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "Qwen2.5 (byte BPE)":    "Qwen/Qwen2.5-0.5B-Instruct",
}
toks = {k: AutoTokenizer.from_pretrained(v) for k, v in names.items()}
samples = ["The unbelievable tokenization of 2026 costs $3.50.",
           "자연어처리는 재미있는 과목입니다."]
for s in samples:
    print("\n" + s) #문장을 하나씩 꺼내서 원문을 출력함
    for k, t in toks.items(): #k :"BERT(WordPiece)같은이름 이고 t:실제 tokenizer 객체"
        ids = t(s, add_special_tokens=False)["input_ids"]
        encoded= t(s, add_special_tokens=False, return_tensors="pt")
        check_sennum=encoded.get("token_type_ids")
        print(f"  {k:<20} {len(ids):>3} tokens  {t.convert_ids_to_tokens(ids)[:12]}")
        print(check_sennum)
        


The unbelievable tokenization of 2026 costs $3.50.
  BERT (WordPiece)      13 tokens  ['the', 'unbelievable', 'token', '##ization', 'of', '202', '##6', 'costs', '$', '3', '.', '50']
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
  GPT-2 (byte BPE)      13 tokens  ['The', 'Ġunbelievable', 'Ġtoken', 'ization', 'Ġof', 'Ġ20', '26', 'Ġcosts', 'Ġ$', '3', '.', '50']
None
  SmolLM2 (byte BPE)    18 tokens  ['The', 'Ġunbelie', 'vable', 'Ġtoken', 'ization', 'Ġof', 'Ġ', '2', '0', '2', '6', 'Ġcosts']
None
  Qwen2.5 (byte BPE)    17 tokens  ['The', 'Ġunbelievable', 'Ġtoken', 'ization', 'Ġof', 'Ġ', '2', '0', '2', '6', 'Ġcosts', 'Ġ$']
None

자연어처리는 재미있는 과목입니다.
  BERT (WordPiece)      28 tokens  ['ᄌ', '##ᅡ', '##ᄋ', '##ᅧ', '##ᆫ', '##ᄋ', '##ᅥ', '##ᄎ', '##ᅥ', '##ᄅ', '##ᅵ', '##ᄂ']
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0]])
  GPT-2 (byte BPE)      45 tokens  ['ì', 'ŀ', 'Ĳ', 'ì', 'Ĺ', '°', 'ì', 'ĸ', '´', 'ì', '²', 'ĺ']
None
  SmolLM2 (byte BPE)

In [21]:
for k, t in toks.items():
    print(f"{k:<20} vocab {len(t):>7,}   special: {t.all_special_tokens[:6]}")

BERT (WordPiece)     vocab  30,522   special: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
GPT-2 (byte BPE)     vocab  50,257   special: ['<|endoftext|>']
SmolLM2 (byte BPE)   vocab  49,152   special: ['<|im_start|>', '<|im_end|>', '<|endoftext|>']
Qwen2.5 (byte BPE)   vocab 151,665   special: ['<|im_end|>', '<|endoftext|>', '<|im_start|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>']


**Question.** Why does the Korean sentence need many more tokens with GPT-2 than with Qwen2.5?
(Hint: Lecture 2 — what does the BPE vocabulary learn from?)

## 3. Batches: padding, truncation, attention mask

In [ ]:
batch = bert_tok(["A short sentence.",
                  "A much longer sentence that needs more tokens than the first one."],
                 padding=True, truncation=True, max_length=16, return_tensors="pt")

#padding 부족한 경우 attention mask부분을 다 0으로 채워 넣음 truncation 은 너무길면 잘라낸다 
print(batch["input_ids"])
print(batch["attention_mask"])       # 0 = padding: attention ignores these positions
print("pad token:", bert_tok.pad_token, bert_tok.pad_token_id)

tensor([[  101,  1037,  2460,  6251,  1012,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1037,  2172,  2936,  6251,  2008,  3791,  2062, 19204,  2015,
          2084,  1996,  2034,  2028,  1012,   102]])
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
pad token: [PAD] 0


## 4. Models and heads

`AutoModel` returns hidden states only. `AutoModelFor<Task>` adds a task head on top of the same body.

In [ ]:
#여기서부터 다시 
import torch
from transformers import AutoModel, AutoModelForMaskedLM

#BERT가 문장을 처리하면서 만든 각층의 hidden state 와 attention weight를 직접 관찰하는 코드 

body = AutoModel.from_pretrained("google-bert/bert-base-uncased",
                                 attn_implementation="eager") 
#eager란 self- attention을 계산하는 방법을 지정하는 옵션
# 일반적인 pytorch 연산으로 계산하라는 뜻 즉 attention weight을 출력하라고 하는 모드임
#   # fused attention kernels do not return weights
inputs = bert_tok("Seoul is the capital of South Korea.", return_tensors="pt")
with torch.no_grad():
    out = body(**inputs, output_hidden_states=True, output_attentions=True)

print(out.last_hidden_state.shape)          # (batch, seq_len, hidden_size)
print(len(out.hidden_states), "hidden states (embeddings + one per layer)")
print(out.attentions[0].shape)              # (batch, heads, seq_len, seq_len)
print(body.config.num_hidden_layers, body.config.hidden_size, body.config.num_attention_heads)

W1029 23:25:47.199000 4111 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7506.87it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 10, 768])
13 hidden states (embeddings + one per layer)
torch.Size([1, 12, 10, 10])
12 768 12


Fill-mask by hand: the MLM head gives a distribution over the vocabulary at the `[MASK]` position.

In [ ]:
mlm = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")
inputs = bert_tok("Seoul is the [MASK] of South Korea.", return_tensors="pt")
with torch.no_grad():
    logits = mlm(**inputs).logits                    # (1, seq_len, vocab_size)
mask_pos = (inputs["input_ids"][0] == bert_tok.mask_token_id).nonzero().item()
probs = torch.softmax(logits[0, mask_pos], dim=-1)
top = probs.topk(5)
for p, i in zip(top.values, top.indices):
    print(f"{bert_tok.decode([int(i)]):>10}  {p:.3f}")

## 5. A causal language model and its loss

For `AutoModelForCausalLM`, passing `labels` makes the model compute the next-token loss itself.
The labels are the input ids; the model shifts them by one position internally
(the same as `x` and `y` in Homework 2's `get_batch`).

In [ ]:
from transformers import AutoModelForCausalLM

name = "HuggingFaceTB/SmolLM2-360M"                  # the base (pre-trained only) model
tok = AutoTokenizer.from_pretrained(name)
lm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)

n_params = sum(p.numel() for p in lm.parameters())
print(f"{n_params/1e6:.0f}M parameters, {n_params*4/1e9:.2f} GB in float32, {n_params*2/1e9:.2f} GB in bfloat16")

inputs = tok("The capital of South Korea is Seoul.", return_tensors="pt")
with torch.no_grad():
    out = lm(**inputs, labels=inputs["input_ids"])
print("logits:", out.logits.shape)                   # (1, seq_len, vocab_size)
print("loss:", out.loss.item(), "perplexity:", torch.exp(out.loss).item())

Check that the built-in loss is the shifted cross-entropy:

In [ ]:
import torch.nn.functional as F
logits, ids = out.logits[0], inputs["input_ids"][0]
manual = F.cross_entropy(logits[:-1], ids[1:])        # position t predicts token t+1
print(manual.item(), out.loss.item())

**Label −100.** Positions whose label is −100 are ignored by the loss.
This is how SFT puts the loss only on the response (Lecture 7, slide 12). Notebook 3 builds such labels
from a chat template.

In [ ]:
labels = inputs["input_ids"].clone()
labels[0, :4] = -100                                  # ignore the first four positions
with torch.no_grad():
    print(lm(**inputs, labels=labels).loss.item())
print(F.cross_entropy(logits[:-1], labels[0, 1:], ignore_index=-100).item())

## 6. Save and load locally

`save_pretrained` writes the same files that are on the Hub; `from_pretrained` accepts a local folder.

In [ ]:
lm.save_pretrained("my-smollm2"); tok.save_pretrained("my-smollm2")
import os; print(sorted(os.listdir("my-smollm2")))
reloaded = AutoModelForCausalLM.from_pretrained("my-smollm2")

## Try it

1. Tokenize a sentence from your own field (Korean or English) with all four tokenizers. Which one is shortest?
2. Mask a different word in Section 4. When does BERT get it wrong?
3. Compute the loss of SmolLM2 on a grammatical sentence and on the same words shuffled. Which perplexity is higher, and why?